# 00 — Data Acquisition

## 1. Introduction

Darb is an intelligent advisory platform for entrepreneurs in the Kingdom of Saudi Arabia. One of its components
is a survival prediction model which, given a founder's business idea, predicts whether the business would
survive its first year. This series of notebooks documents the development of that model, from the acquisition
of the raw data to the training of the final model.

### 1.1 Research Question

> **Given a founder's business idea, can we predict whether the business will still be operating one year after
> its establishment?**

The research question is examined through two sub-questions:

- **RQ1.** To what extent do the characteristics of the idea (economic sector, region, city, legal form, planned
  capital and e-commerce activity) contribute to predicting first-year survival?
- **RQ2.** To what extent do the conditions of the market the idea would enter (sectoral GDP growth, consumer card
  spending and market crowding) contribute to predicting first-year survival?

### 1.2 Prediction Target

For a founder's business idea, the model predicts one of two outcomes, together with its probability:

- **Survive:** the business is still operating one year after its establishment.
- **Fail:** the business ceases operating within its first year.

### 1.3 Measurement of the Target

An idea cannot be observed before it is implemented. The model therefore learns from businesses already
established in Saudi Arabia, using the Ministry of Commerce records of active commercial registrations:

- A business whose registration is **active four quarters after its founding quarter** is recorded as having
  **survived**.
- A business whose registration is **not active at that point** is recorded as having **failed**.
- A business absent at that point is recorded as having failed, **even if it reappears in a later record** (for
  example, after a temporary suspension), since it was not active at the end of its first year.

The model learns how the characteristics of a business and the market conditions at its founding relate to its
survival. It then applies this relationship to the founder's idea, using the most recent market conditions
available. Because survival is observed through registration status, the target reflects the continuity of a
registration rather than the commercial performance of the business. This limitation is acknowledged throughout
the study.

### 1.4 Data Sources

| Source | Dataset | Period | Purpose in the study |
|---|---|---|---|
| Ministry of Commerce (MC) | Quarterly records of active commercial registrations | 2023 Q1 – 2026 Q3 | Business characteristics, survival outcome, market crowding |
| General Authority for Statistics (GASTAT) | Gross domestic product by economic activity | 2010 Q1 – 2026 Q2 | Sectoral economic growth |
| Saudi Central Bank (SAMA), published on DataSaudi | Annual point-of-sale transactions by economic sector | 2016 – 2025 | Consumer demand by sector |
| Saudi Central Bank (SAMA), published on DataSaudi | Annual point-of-sale transactions by major city | To be acquired | Consumer demand by city |
| GASTAT | National Classification of Economic Activities (SSIC) | September 2026 edition | Sector codes and names |
| Ministry of Commerce | Registered economic activities of active commercial registrations (ارتباط السجلات التجارية القائمة مع التصنيف الوطني للأنشطة الاقتصادية) | 2023, 2024 and 2025 | Reference for inferring the economic sector from the business name; not used as a model input |
| GASTAT | Operating revenues and expenses of small and medium enterprises | 2019 – 2024 | Evaluated as a candidate indicator; excluded from the final model (Notebook 05) |

The annual sector figures are sufficient because each business is described by the complete calendar year
preceding its founding; the 2025 figures, the most recent complete year, therefore describe the market for an
idea evaluated in 2026.

### 1.5 Methodology

The study is a **retrospective cohort study with out-of-time evaluation**. Businesses are grouped into cohorts by
the quarter in which they were founded. Each business is described by the information available at its
registration and by the market conditions that preceded its founding, and is observed for one year. Models are
trained on earlier cohorts and evaluated on later ones, so that every reported result reflects performance on
businesses founded after those used for training.

#### 1.5.1 Pipeline

The study follows a sequential pipeline. Each notebook performs one stage and stores its results, which the
subsequent notebook loads.

| Notebook | Stage |
|---|---|
| 00 | Data acquisition |
| 01 | Exploratory data analysis of the raw data |
| 02 | Preprocessing of business records and construction of the survival target |
| 03 | Preprocessing of the economic indicators |
| 04 | Exploratory data analysis of the cleaned data |
| 05 | Feature engineering and temporal data partitioning |
| 06 | Baseline models |
| 07 | Hyperparameter tuning |
| 08 | Final model training |

#### 1.5.2 Unit of Analysis

Each observation is a business, identified by its main commercial registration. Branch registrations describe an
existing business rather than a new one, and are therefore not treated as separate observations.

#### 1.5.3 Cohorts

The founding quarter of every business is recorded as a variable, so that every cohort remains identifiable
throughout the study.

#### 1.5.4 Data Partitioning

The cohorts are partitioned chronologically, so that the model is always assessed on businesses founded after
those it learned from:

1. **Training cohorts:** the earliest cohorts, from which the models learn.
2. **Validation cohorts:** the cohorts that follow, used to compare models, select features and tune
   hyperparameters.
3. **Test cohort:** the most recent cohort, held out entirely and used once for the final evaluation.

No cohort contributes to more than one partition.

#### 1.5.5 Models

Candidate models are compared against baselines of increasing strength:

1. The overall survival rate of the training cohorts.
2. The historical survival rate of businesses in the same sector and city.
3. A regularised logistic regression.

The candidate models are gradient-boosted decision trees (LightGBM and XGBoost). No resampling of the minority
class is applied, so that the predicted probabilities remain interpretable.

#### 1.5.6 Evaluation Metrics

Failure is treated as the positive class.

- **Discrimination** is measured by the area under the precision–recall curve (PR-AUC).
- **Calibration** is measured by the Brier score and a calibration curve.
- **Recall and precision** are reported as secondary measures.

A group of features is retained only if it increases PR-AUC on the validation cohorts, and the 95% bootstrap
confidence interval of that increase excludes zero. This rule is fixed before any model is trained.

#### 1.5.7 Principles

1. Raw data are never modified.
2. No record is discarded without a stated justification.
3. Every market indicator is measured strictly before the founding quarter of each business, so that no
   information from the future is used.
4. A feature group is retained only if it produces a measured improvement in predictive performance, according
   to the rule in Section 1.5.6.

### 1.6 Legal and Regulatory References

The treatment of the Ministry of Commerce records relies on the following official sources:

| Source | Issued by | Relevance to the study |
|---|---|---|
| [Companies Law (نظام الشركات)](https://mc.gov.sa/ar/Regulations/pages/details.aspx?lawId=ea0fc797-4127-4667-962d-aec400ee9f72), Royal Decree M/132 dated 1/12/1443 AH, in force from 19 January 2023 | Ministry of Commerce; official text by the Bureau of Experts at the Council of Ministers | Article 4 defines the five forms of companies, used as the standard spelling of the legal form. Article 2 permits one-person and non-profit companies, and Article 216 defines holding companies, which are therefore variants of a form rather than separate forms. Article 220 refers to sole proprietorships (المؤسسات الفردية). Part 11 (Articles 235–241) governs branches of foreign companies. |
| [Commercial Register Law (نظام السجل التجاري) and its Implementing Regulation](https://mc.gov.sa/ar/Regulations/pages/details.aspx?lawId=3d536f10-11ce-49db-9153-b204016f4fca), Royal Decree M/83 dated 19/3/1446 AH | Ministry of Commerce | Article 15 of the Law provides that a registration is suspended if its annual confirmation is not submitted within 90 days, for a period of one year, after which it is reinstated or struck off; a suspended registration is not active. Article 6 of the Regulation provides that a trader is registered once, regardless of the number of activities and branches. |
| Implementing Regulation of the Commercial Register Law (same source as above) | Ministry of Commerce | Article 3 requires every registration request to state the type of activity, and Article 15 includes the trader's activities among the registry's data. Article 9 of the Law permits a trader to register different, unrelated activities. These provisions establish the registered activities as the official record of each business's activity. |
| [Approval of the unified number starting with 7](https://rulebook.sama.gov.sa/en/node/10194), Council of Ministers Resolution No. 225 dated 6/5/1439 AH | Saudi Central Bank Rulebook, circular dated 11/2/1442 AH | Defines the unified number of non-governmental establishments: ten digits starting with 7, issued by the Ministry of Commerce for commercial registrations. It is the registration number used to follow each business across the quarterly records. |
| [Announcement of the entry into force of the Commercial Register and Trade Names Laws](https://mc.gov.sa/ar/mediacenter/News/Pages/31-03-25-01.aspx) | Ministry of Commerce, 2 Shawwal 1446 AH | The laws take effect on 5 Shawwal 1446 AH; the commercial registration number becomes the unified number starting with 7, and a trader holds a single registration. |
| [Announcement on the application of the new Companies Law](https://mc.gov.sa/ar/mediacenter/News/Pages/04-01-23-01.aspx) | Ministry of Commerce and Capital Market Authority, 4 January 2023 | Confirms the date of entry into force and the two-year period granted to existing companies to comply with the new law. |

## 2. Purpose of This Notebook

This notebook acquires every raw dataset used in the study and stores an exact copy of each in a fast-loading
format. All subsequent notebooks read only these copies. This guarantees that every stage operates on identical
data, and avoids repeatedly reading the original spreadsheet files, which is slow.

## 3. Environment Configuration

This cell identifies the location of the data. When executed locally, the data are read from a local copy of the
project's shared data folder. When executed on Google Colab, the shared folder is accessed directly through
Google Drive.

In [1]:
import os
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = Path('/content/drive/MyDrive/Darb DataSets')   # a shortcut to the team folder in your My Drive
    SRC_DIR = Path('/content')                                # upload the files of ml/src here
    %pip install -q python-calamine lightgbm xgboost shap
else:
    DATA_DIR = Path(os.environ.get('DARB_DATA', Path.home() / 'darb-data'))   # local copy of Darb DataSets
    SRC_DIR = Path.cwd().parent / 'src'

PIPE = DATA_DIR / 'pipeline'          # every notebook saves its results here (never in Git)
PIPE.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(SRC_DIR))


def need(path, notebook):
    """Stops with a clear message when a file from an earlier notebook is missing."""
    if not Path(path).exists():
        raise FileNotFoundError(f'{Path(path).name} not found: run {notebook} first.')
    return Path(path)


print('Colab' if IN_COLAB else 'Local', '| data:', DATA_DIR, '| results:', PIPE)

Local | data: /Users/halaalsabbar/darb-data | results: /Users/halaalsabbar/darb-data/pipeline


## 4. Inventory of Raw Datasets

This cell lists the raw files required by the study and locates each one within the data folder. Any missing file
is reported.

In [2]:
from mc_files import MC_FILES

RAW_FILES = (
    [('mc', name) for _, name in MC_FILES]
    + [('gastat', 'gastat_gdp_2026-10-21T09_14_09.657Z.csv'),
       ('gastat', 'gastat_sme_operating_expenses_revenues_2026-10-21T09_39_24.273Z.csv'),
       ('gastat', 'National Classification for the Economic Activities(SSIC)_15 Sep_2026_AR.xlsx'),
       ('sama', 'sama_pos_sectors_2026-11-15T19_39_54.944Z.csv'),
       ('sama', 'point-of-sale-transactions-by-sector-and-city.xlsx'),
       ('sama', 'sama_pos_weekly_2025-06_to_2026-11.csv')]
    + [('activity', f'CR_ISISC_2023_PART{i}.xlsx') for i in range(1, 11)]
    + [('activity', f'CR_ISISC_2024_PART{i}.xlsx') for i in range(1, 14)]
    + [('activity', 'CR_ISIC_2025.csv')]
)

# Where each file is: searched once in the whole data folder (the pipeline folder itself is skipped)
found = {}
for path in DATA_DIR.rglob('*'):
    if path.is_file() and PIPE not in path.parents and not path.name.startswith('~$'):
        found.setdefault(path.name, path)

missing = [name for _, name in RAW_FILES if name not in found]
print(f'{len(RAW_FILES)} raw files | found: {len(RAW_FILES) - len(missing)} | missing: {len(missing)}')
for name in missing:
    print('   MISSING:', name)

60 raw files | found: 60 | missing: 0


**Result.** All 60 raw files required by the study were found in the data folder; none is missing.

**Interpretation.** The acquisition can proceed with the complete set of inputs: 30 Ministry of Commerce quarterly
files, 24 files of registered activities (10 for 2023, 13 for 2024 and one for 2025), three files of the General
Authority for Statistics and three files of the Saudi Central Bank.

## 5. Storage of a Fast-Loading Copy

Each raw file is read once and stored as a fast-loading copy (in the Parquet format, which stores data column by
column and loads in seconds). All values are preserved as text, so that identifiers
and codes are not altered (for example, leading zeros are retained). The classification file of the General
Authority for Statistics begins with title rows and is therefore stored without a header; its header is
identified when the file is processed. A file that has already been converted is not processed again. Each copy is first
written under a temporary name and renamed only when complete, so that an interrupted conversion cannot produce a
partial file.

In [3]:
import shutil
import time

import pandas as pd

RAW_DIR = PIPE / '00_raw'
OLD_COPIES = DATA_DIR / 'saved' / 'mc_excel_copies'      # copies made by the experiment notebooks
NO_HEADER = {'National Classification for the Economic Activities(SSIC)_15 Sep_2026_AR.xlsx'}


def save_large_csv(path, target):
    # A very large CSV (the 2025 activity file, 4.5 GB) is copied block by block, all as text, to limit memory.
    import pyarrow as pa
    import pyarrow.csv as pacsv
    import pyarrow.parquet as pq
    with open(path, encoding='utf-8-sig') as f:
        names = f.readline().strip().split(',')
    reader = pacsv.open_csv(path, convert_options=pacsv.ConvertOptions(column_types={n: pa.string() for n in names},
                                                                      strings_can_be_null=True))
    with pq.ParquetWriter(target, reader.schema) as writer:
        for batch in reader:
            writer.write_batch(batch)


def read_raw(path):
    header = None if path.name in NO_HEADER else 0
    if path.suffix == '.csv':
        return pd.read_csv(path, dtype=str, header=header, keep_default_na=False, na_values=[''])
    return pd.read_excel(path, engine='calamine', dtype=str, header=header)


for group, name in RAW_FILES:
    copy = RAW_DIR / group / f'{name}.parquet'
    if copy.exists():
        print(f'already saved   {name}')
        continue
    if name not in found:
        continue
    copy.parent.mkdir(parents=True, exist_ok=True)
    started = time.time()
    if (OLD_COPIES / f'{name}.parquet').exists():
        shutil.copyfile(OLD_COPIES / f'{name}.parquet', copy.with_suffix('.part'))
        how = 'copied from saved/mc_excel_copies'
    elif found[name].suffix == '.csv' and found[name].stat().st_size > 1e9:
        save_large_csv(found[name], copy.with_suffix('.part'))
        how = 'read from the raw file, block by block'
    else:
        table = read_raw(found[name])
        table.columns = [str(c) for c in table.columns]    # parquet needs text column names
        table.to_parquet(copy.with_suffix('.part'), index=False)
        how = 'read from the raw file'
    copy.with_suffix('.part').rename(copy)
    print(f'saved           {name}  ({how}, {time.time() - started:.0f} s)')

already saved   CR_2023-01q-active-crs_PART1.xlsx
already saved   CR_2023-01q-active-crs_PART2.xlsx
already saved   CR_2023-02q-active-crs_PART1.xlsx
already saved   CR_2023-02q-active-crs_PART2.xlsx
already saved   CR_2023-03q-active-crs_PART1.xlsx
already saved   CR_2023-03q-active-crs_PART2.xlsx
already saved   CR_2023-04q-active-crs_PART1.xlsx
already saved   CR_2023-04q-active-crs_PART2.xlsx
already saved   CR_2024-01q-active-crs_PART1.xlsx
already saved   CR_2024-01q-active-crs_PART2.xlsx
already saved   CR_2024-02q-active-crs_PART1.xlsx
already saved   CR_2024-02q-active-crs_PART2.xlsx
already saved   CR_2024-03q-active-crs_PART1-2.xlsx
already saved   CR_2024-03q-active-crs_PART2-2.xlsx
already saved   CR_2024-04q-active-crs_PART1.xlsx
already saved   CR_2024-04q-active-crs_PART2.xlsx
already saved   CR_2025-01q-active-crs_PART1.xlsx
already saved   CR_2025-01q-active-crs_PART2.xlsx
already saved   2025-02q-active-crs_PART1-2.xlsx
already saved   2025-02q-active-crs_PART2.xlsx


**Result.** The 58 files converted in an earlier run were not processed again ("already saved"). The two Saudi
Central Bank files added to the inventory (point-of-sale transactions by sector and city, and weekly point-of-sale
transactions) were converted for the first time, each in under a second.

**Interpretation.** Every raw file now has a fast-loading copy, and later notebooks never need to read the original
spreadsheets again. Re-running this notebook only converts files that are new or have been removed.

## 6. Summary of Acquired Data

This cell reports, for each dataset, the number of records and columns, the size of the stored copy and the
location of the original file. The summary is saved as `manifest.csv`, which records the exact inputs of the
study, and a copy is kept in the repository at `ml/reports/00_manifest.csv`.

In [4]:
import pyarrow.parquet as pq

rows = []
for group, name in RAW_FILES:
    copy = RAW_DIR / group / f'{name}.parquet'
    if not copy.exists():
        rows.append({'group': group, 'file': name, 'rows': None, 'columns': None, 'copy MB': None, 'raw file': 'MISSING'})
        continue
    meta = pq.ParquetFile(copy).metadata
    rows.append({'group': group, 'file': name, 'rows': meta.num_rows, 'columns': meta.num_columns,
                 'copy MB': round(copy.stat().st_size / 1e6, 1),
                 'raw file': str(found[name].relative_to(DATA_DIR)) if name in found else '(copy only)'})
manifest = pd.DataFrame(rows)
manifest.to_csv(RAW_DIR / 'manifest.csv', index=False, encoding='utf-8-sig')
print(f'Rows in all MC files: {manifest.loc[manifest["group"] == "mc", "rows"].sum():,.0f}')
manifest

Rows in all MC files: 19,749,510


,group,file,rows,columns,copy MB,raw file
0,mc,CR_2023-01q-active-crs_PART1.xlsx,1026158,11,51.8,MC/CR_2023-01q-active-crs_PART1.xlsx
1,mc,CR_2023-01q-active-crs_PART2.xlsx,300870,11,14.7,MC/CR_2023-01q-active-crs_PART2.xlsx
2,mc,CR_2023-02q-active-crs_PART1.xlsx,1026112,11,51.5,MC/CR_2023-02q-active-crs_PART1.xlsx
3,mc,CR_2023-02q-active-crs_PART2.xlsx,314957,11,15.3,MC/CR_2023-02q-active-crs_PART2.xlsx
4,mc,CR_2023-03q-active-crs_PART1.xlsx,10449,11,0.5,MC/CR_2023-03q-active-crs_PART1.xlsx
5,mc,CR_2023-03q-active-crs_PART2.xlsx,347042,11,16.8,MC/CR_2023-03q-active-crs_PART2.xlsx
6,mc,CR_2023-04q-active-crs_PART1.xlsx,1025540,11,51.1,MC/CR_2023-04q-active-crs_PART1.xlsx
7,mc,CR_2023-04q-active-crs_PART2.xlsx,373870,11,18.1,MC/CR_2023-04q-active-crs_PART2.xlsx
8,mc,CR_2024-01q-active-crs_PART1.xlsx,1024896,11,50.7,MC/CR_2024-01q-active-crs_PART1.xlsx
9,mc,CR_2024-01q-active-crs_PART2.xlsx,428983,11,21.1,MC/CR_2024-01q-active-crs_PART2.xlsx


**Result.** The thirty Ministry of Commerce files contain 19,749,510 records in total. Their sizes differ markedly:
for example, part 1 of 2023 Q3 contains 10,449 records and part 1 of 2025 Q3 contains 109,541 records with only
five columns, whereas the other parts contain between 146,666 and 1,026,158 records.

**Interpretation.** The manifest records the exact inputs of the study and is kept in the repository as
`ml/reports/00_manifest.csv`. The differences in size and structure between files are examined in Notebook 01.